In [ ]:
# ---------- 1. 路径与数据（Kaggle 适配）----------
MODEL_PATH = "bert-base-chinese"          # 直接从 HuggingFace 下载
# 请把 train.txt / test.txt / dev.txt 上传为 Kaggle 数据集，然后改下面的目录名
DATA_DIR = "/kaggle/input/datasets/wubarry/toumanfen-news/"
TRAIN_FILE = f"{DATA_DIR}/train.txt"
TEST_FILE  = f"{DATA_DIR}/test.txt"
DEV_FILE = f"{DATA_DIR}/dev.txt"
SAVED_HEADS = "/kaggle/working/nested_heads.pt"   # Kaggle 工作区可写

In [ ]:
!pip install -q tqdm
import os
# Kaggle 镜像有时会带 TensorFlow，禁用以避免 Keras 3 与 Transformers 不兼容
os.environ['TRANSFORMERS_NO_TF'] = '1'

# 投满分新闻分类：冻结 BERT，输入侧/输出侧各套一个单层网络（套娃方案），只训这两层
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
from transformers import BertTokenizerFast, BertModel

train_df = pd.read_csv(TRAIN_FILE, sep='\t', header=None)
train_df.columns = ['sentence', 'label']
test_df = pd.read_csv(TEST_FILE, sep='\t', header=None)
test_df.columns = ['sentence', 'label']

# 只取十分之一样本用于训练/评测，加快速度（保持类别分布，随机种子固定可复现）
train_df = train_df.sample(frac=0.1, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=0.1, random_state=42).reset_index(drop=True)
num_labels = int(train_df['label'].nunique())   # 投满分数据集为 0~9 共 10 类
MAX_LEN = 64

# ---------- 2. 分词器与数据集 ----------
tokenizer = BertTokenizerFast.from_pretrained(MODEL_PATH)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN):
        self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

train_ds = NewsDataset(train_df['sentence'], train_df['label'])
test_ds  = NewsDataset(test_df['sentence'], test_df['label'])

# ---------- 3. 套娃网络：冻结 BERT，两端各加一个单层网络 ----------
bert = BertModel.from_pretrained(MODEL_PATH)
H = bert.config.hidden_size

class NeoBert(nn.Module):
    def __init__(self, bert, num_labels, hidden_size):
        super().__init__()
        self.bert = bert
        self.post_net = nn.Linear(hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        emb = self.bert.get_input_embeddings()(input_ids)
        out = self.bert(inputs_embeds=emb, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.post_net(cls)
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
            return {'loss': loss, 'logits': logits}
        return logits

model = NeoBert(bert, num_labels, H)
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"可训练参数数量：{sum(p.numel() for p in trainable):,}")

# ---------- 4. 训练（只更新套娃网络）----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model.to(device)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)
optimizer = torch.optim.AdamW(trainable, lr=2e-5)

EPOCHS = 2
model.train()
for epoch in range(EPOCHS):
    running_loss, total, correct = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        ttype = batch.get('token_type_ids')
        if ttype is not None:
            ttype = ttype.to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        out = model(input_ids, attn, ttype, labels)
        loss = out['loss']
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
        total += labels.size(0)
        correct += (out['logits'].argmax(-1) == labels).sum().item()
        pbar.set_postfix(loss=f"{running_loss/total:.4f}", acc=f"{correct/total:.2%}")
    print(f"Epoch {epoch+1}/{EPOCHS}  损失={running_loss/total:.4f}  训练准确率={correct/total:.2%}")

    # ---------- 5. 评测（四项指标）----------
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            ttype = batch.get('token_type_ids')
            if ttype is not None:
                ttype = ttype.to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attn, ttype)
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    model.train()

    acc = accuracy_score(all_labels, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    print(f"模型准确率：{acc:.2%}")
    print(f"模型精确率：{p:.2%}，模型召回率：{r:.2%}，模型F1值：{f1:.2%}")
    
    # 保存套娃网络
    torch.save({'BERT': model.bert.state_dict(), 
                'post_net': model.post_net.state_dict()}, SAVED_HEADS)

# ---------- 6. 预测函数 ----------
def predict(title):
    model.eval()
    enc = tokenizer([title], truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    attn = enc['attention_mask'].to(device)
    ttype = enc.get('token_type_ids')
    if ttype is not None:
        ttype = ttype.to(device)
    with torch.no_grad():
        logits = model(input_ids, attn, ttype)
    return int(logits.argmax(-1).item())

In [ ]:
import os
os.environ['TRANSFORMERS_NO_TF'] = '1'

# 在上一 cell 保存的套娃网络基础上继续训练 100 epoch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import BertTokenizerFast, BertModel
from tqdm import tqdm

MODEL_PATH = "bert-base-chinese"
DATA_DIR = "/kaggle/input/toumanfen-data"
TRAIN_FILE = f"{DATA_DIR}/train.txt"
SAVED_HEADS = "/kaggle/working/nested_heads.pt"
MAX_LEN = 64

# ---------- 1. 读取原始全量数据（每个 epoch 随机抽 1024）----------
full_df = pd.read_csv(TRAIN_FILE, sep='\t', header=None)
full_df.columns = ['sentence', 'label']
num_labels = int(full_df['label'].nunique())

# ---------- 2. 分词器与数据集 ----------
tokenizer = BertTokenizerFast.from_pretrained(MODEL_PATH)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN):
        self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

# ---------- 3. 重建套娃网络并加载权重 ----------
bert = BertModel.from_pretrained(MODEL_PATH)
H = bert.config.hidden_size

class NestedBert(nn.Module):
    def __init__(self, bert, num_labels, hidden_size):
        super().__init__()
        self.bert = bert
        for p in self.bert.parameters():
            p.requires_grad = False
        self.bert.eval()
        self.pre_net = nn.Linear(hidden_size, hidden_size)
        self.post_net = nn.Linear(hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        emb = self.bert.get_input_embeddings()(input_ids)
        emb = self.pre_net(emb)
        out = self.bert(inputs_embeds=emb, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.post_net(cls)
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
            return {'loss': loss, 'logits': logits}
        return logits

model = NestedBert(bert, num_labels, H)
ckpt = torch.load(SAVED_HEADS, map_location='cpu', weights_only=True)
model.pre_net.load_state_dict(ckpt['pre_net'])
model.post_net.load_state_dict(ckpt['post_net'])
print("已加载上一 cell 保存的套娃网络权重（pre_net / post_net）")

# ---------- 4. 继续训练 100 epoch，每 epoch 随机抽 1024 条 ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=2e-5)

EPOCHS = 100
model.train()
model.bert.eval()
for epoch in range(1, EPOCHS + 1):
    epoch_df = full_df.sample(n=1024, random_state=epoch)
    epoch_ds = NewsDataset(epoch_df['sentence'], epoch_df['label'])
    loader = DataLoader(epoch_ds, batch_size=32, shuffle=True)
    running_loss, total, correct = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        ttype = batch.get('token_type_ids')
        if ttype is not None:
            ttype = ttype.to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        out = model(input_ids, attn, ttype, labels)
        loss = out['loss']
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
        total += labels.size(0)
        correct += (out['logits'].argmax(-1) == labels).sum().item()
        pbar.set_postfix(loss=f"{running_loss/total:.4f}", acc=f"{correct/total:.2%}")
    print(f"Epoch {epoch}/{EPOCHS}  损失={running_loss/total:.4f}  训练准确率={correct/total:.2%}")
    torch.save({'pre_net': model.pre_net.state_dict(), 'post_net': model.post_net.state_dict()}, SAVED_HEADS)

torch.save({'pre_net': model.pre_net.state_dict(), 'post_net': model.post_net.state_dict()}, SAVED_HEADS)
print("已保存继续训练后的套娃网络权重 ->", SAVED_HEADS)



In [ ]:
import os
os.environ['TRANSFORMERS_NO_TF'] = '1'

# 读取保存的套娃网络，用 dev.txt 评测四大指标
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
from transformers import BertTokenizerFast, BertModel

MAX_LEN = 64

# ---------- 1. 分词器与数据集 ----------
tokenizer = BertTokenizerFast.from_pretrained(MODEL_PATH)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN):
        self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

# ---------- 2. 读取 dev 数据 ----------
dev_df = pd.read_csv(DEV_FILE, sep='\t', header=None)
dev_df.columns = ['sentence', 'label']
# 每次随机抽取 1/10 的 dev 作验证，加快评测
dev_df = dev_df.sample(frac=0.1, random_state=None).reset_index(drop=True)
print(f"dev 抽样规模：{len(dev_df)} 条（占总 dev 的 10%）")

# ---------- 3. 重建套娃网络并加载权重 ----------
bert = BertModel.from_pretrained(MODEL_PATH)
H = bert.config.hidden_size

class NeoBert(nn.Module):
    def __init__(self, bert, num_labels, hidden_size):
        super().__init__()
        self.bert = bert
        for p in self.bert.parameters():
            p.requires_grad = False
        self.bert.eval()
        self.post_net = nn.Linear(hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        emb = self.bert.get_input_embeddings()(input_ids)
        out = self.bert(inputs_embeds=emb, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.post_net(cls)
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
            return {'loss': loss, 'logits': logits}
        return logits

ckpt = torch.load(SAVED_HEADS, map_location='cpu', weights_only=True)
num_labels = ckpt['post_net']['weight'].shape[0]

model = NeoBert(bert, num_labels, H)
model.post_net.load_state_dict(ckpt['post_net'])
print(f"已加载网络权重，类别数 = {num_labels}")

# ---------- 4. 在 dev 集上评测四大指标 ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
dev_ds = NewsDataset(dev_df['sentence'], dev_df['label'])
dev_loader = DataLoader(dev_ds, batch_size=64, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(dev_loader, desc="dev评测", unit="batch"):
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        ttype = batch.get('token_type_ids')
        if ttype is not None:
            ttype = ttype.to(device)
        labels = batch['labels'].to(device)
        logits = model(input_ids, attn, ttype)
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

acc = accuracy_score(all_labels, all_preds)
p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
print(f"dev 集准确率：{acc:.2%}")
print(f"dev 集精确率：{p:.2%}，dev 集召回率：{r:.2%}，dev 集F1值：{f1:.2%}")

In [ ]:
import os
os.environ['TRANSFORMERS_NO_TF'] = '1'

# ============ 全量微调 BERT 的 uint4(4位)权重量化 + 量化前后四大指标对比 ============
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
from transformers import BertTokenizerFast, BertModel, BertConfig

MODEL_PATH  = "bert-base-chinese"
DEV_FILE    = "/kaggle/input/datasets/wubarry/toumanfen-news/dev.txt"
SAVED_HEADS = "/kaggle/working/nested_heads.pt"           # 第2块保存的全量微调权重
QUANT_HEADS = "/kaggle/working/nested_heads_quantized.pt" # 量化后保存位置
MAX_LEN = 64

# ---------- 1. uint4(权重-only)量化 Linear：权重打包进 uint8(每字节存2个4位权重) ----------
class QuantizedLinear4bit(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.register_buffer('packed', torch.zeros(0, dtype=torch.uint8))  # 4位打包权重
        self.register_buffer('scale', torch.ones(1))
        self.register_buffer('zero_point', torch.zeros(1))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)

    @classmethod
    def from_float(cls, linear):
        # 全程在 CPU 上计算量化/打包，避免 CUDA/CPU 设备不一致
        w = linear.weight.data.detach().float().cpu().clone()
        w_min, w_max = float(w.min()), float(w.max())
        qmin, qmax = 0.0, 15.0
        scale = (w_max - w_min) / (qmax - qmin)
        if scale <= 0:
            scale = 1e-8
        zp = int(round(max(qmin, min(qmax, qmin - w_min / scale))))
        q = torch.clamp(torch.round(w / scale) + zp, 0, 15).to(torch.int32)  # 4位量化值 0..15
        flat_q = q.flatten()
        low = flat_q[0::2].to(torch.uint8)   # 偶数索引
        high = flat_q[1::2].to(torch.uint8)  # 奇数索引
        packed = torch.zeros((flat_q.numel() + 1) // 2, dtype=torch.uint8)
        packed[:low.numel()]  |= low
        packed[:high.numel()] |= (high << 4)  # 高4位放高位
        m = cls(linear.in_features, linear.out_features, linear.bias is not None)
        m.register_buffer('packed', packed)   # 覆盖占位 buffer
        m.scale.copy_(torch.tensor([scale], dtype=torch.float32))
        m.zero_point.copy_(torch.tensor([float(zp)], dtype=torch.float32))
        if linear.bias is not None:
            m.bias.data.copy_(linear.bias.data.detach().float())
        return m

    def forward(self, x):
        M = self.in_features * self.out_features
        packed = self.packed.to(device=x.device)
        low = (packed & 0x0F).to(torch.int32)
        high = ((packed >> 4) & 0x0F).to(torch.int32)
        flat_q = torch.empty(M, dtype=torch.int32, device=x.device)
        flat_q[0::2] = low
        flat_q[1::2] = high[:M // 2]           # 奇数长度时截断，与打包配对
        scale = self.scale.to(device=x.device)
        zp = self.zero_point.to(device=x.device)
        w = (flat_q.to(x.dtype) - zp) * scale   # 反量化
        w = w.reshape(self.out_features, self.in_features)
        return F.linear(x, w, self.bias)

def quantize_linears(module):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, QuantizedLinear4bit.from_float(child))
        else:
            quantize_linears(child)

# ---------- 2. 重建 NeoBert 并加载全量微调权重 ----------
tokenizer = BertTokenizerFast.from_pretrained(MODEL_PATH)
H = BertConfig.from_pretrained(MODEL_PATH).hidden_size

class NeoBert(nn.Module):
    def __init__(self, bert, num_labels, hidden_size):
        super().__init__()
        self.bert = bert
        self.post_net = nn.Linear(hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        emb = self.bert.get_input_embeddings()(input_ids)
        out = self.bert(inputs_embeds=emb, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.post_net(cls)
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
            return {'loss': loss, 'logits': logits}
        return logits

ckpt = torch.load(SAVED_HEADS, map_location='cpu', weights_only=True)
num_labels = ckpt['post_net']['weight'].shape[0]
bert = BertModel.from_pretrained(MODEL_PATH)
model = NeoBert(bert, num_labels, H)
if 'BERT' in ckpt:
    model.bert.load_state_dict(ckpt['BERT'])
    model.post_net.load_state_dict(ckpt['post_net'])
    print(f"已加载全量微调权重（BERT + post_net），类别数 = {num_labels}")
else:
    model.post_net.load_state_dict(ckpt['post_net'])
    print("警告：ckpt 无 'BERT' 键，post_net 挂在预训练 BERT 上（非全量微调结果）")

# ---------- 3. dev 数据集（抽 1/10） ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN):
        self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

dev_df = pd.read_csv(DEV_FILE, sep='\t', header=None)
dev_df.columns = ['sentence', 'label']
dev_df = dev_df.sample(frac=0.1, random_state=None).reset_index(drop=True)
print(f"dev 抽样规模：{len(dev_df)} 条（占总 dev 的 10%）")
dev_ds = NewsDataset(dev_df['sentence'], dev_df['label'])
dev_loader = DataLoader(dev_ds, batch_size=64, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)
model.to(device)
model.eval()

def eval_model(model, loader, device):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="dev评测", unit="batch"):
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            ttype = batch.get('token_type_ids')
            if ttype is not None:
                ttype = ttype.to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attn, ttype)
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    return acc, p, r, f1

# ---------- 4. 量化前评测 ----------
acc0, p0, r0, f10 = eval_model(model, dev_loader, device)
print("量化前(float32)：")
print(f"  准确率：{acc0:.2%}  精确率：{p0:.2%}  召回率：{r0:.2%}  F1：{f10:.2%}")

# ---------- 5. 对所有 nn.Linear 做 uint4 量化（BERT 编码器 + 分类头） ----------
quantize_linears(model)
model.to(device)
model.eval()
n_q = sum(1 for m in model.modules() if isinstance(m, QuantizedLinear4bit))
print(f"已量化 {n_q} 个 Linear 层为 uint4（BERT 编码器 + post_net，每字节存2个权重）")

# ---------- 6. 量化后评测 ----------
acc1, p1, r1, f11 = eval_model(model, dev_loader, device)
print("量化后(uint4)：")
print(f"  准确率：{acc1:.2%}  精确率：{p1:.2%}  召回率：{r1:.2%}  F1：{f11:.2%}")

# ---------- 7. 保存量化模型 + 体积对比 ----------
torch.save({'BERT': model.bert.state_dict(), 'post_net': model.post_net.state_dict()}, QUANT_HEADS)
print(f"已保存 uint4 量化权重 -> {QUANT_HEADS}")
s0 = os.path.getsize(SAVED_HEADS); s1 = os.path.getsize(QUANT_HEADS)
print(f"权重大小：原始 {s0/1024/1024:.1f} MB -> 量化 {s1/1024/1024:.1f} MB（压缩 {s0/s1:.2f}x）")
